In [ ]:
# Environment Setup (Local TITAN X / Colab T4)
# V11 STABLE GROWTH - Auto-detects environment and GPU capacity

import os
import gc
import sys

# Detect environment FIRST (before any other imports)
try:
 from google.colab import drive
 drive.mount('/content/drive')
 print(" Google Drive mounted - Running on Colab")
 IN_COLAB = True
except:
 print(" Running on Local Machine")
 IN_COLAB = False

# Fix Colab's numpy/pandas compatibility issue
if IN_COLAB:
 print(" Fixing Colab package compatibility...")
 # Install compatible versions to avoid numpy.dtype size error
 os.system('pip install numpy==1.26.4 pandas==2.0.3 pydicom -q 2>/dev/null')
 # Force reimport of numpy with correct version
 if 'numpy' in sys.modules:
 del sys.modules['numpy']
 if 'pandas' in sys.modules:
 del sys.modules['pandas']
else:
 # Local: just ensure pydicom is installed
 os.system('pip install pydicom -q 2>/dev/null')

# Now import everything
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from PIL import Image
import warnings
import cv2
from tqdm import tqdm
import json
from datetime import datetime
warnings.filterwarnings('ignore')

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

# Metrics & visualization
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Detect GPU and set memory growth
gpus = tf.config.list_physical_devices('GPU')
GPU_NAME = "CPU"
GPU_MEMORY_GB = 0

if gpus:
 for gpu in gpus:
 tf.config.experimental.set_memory_growth(gpu, True)
 
 # Get GPU info
 gpu_details = tf.config.experimental.get_device_details(gpus[0])
 GPU_NAME = gpu_details.get('device_name', 'Unknown GPU')
 
 # Estimate memory
 if 'TITAN' in GPU_NAME.upper():
 GPU_MEMORY_GB = 12
 GPU_TYPE = "TITAN_X"
 elif 'T4' in GPU_NAME.upper():
 GPU_MEMORY_GB = 16
 GPU_TYPE = "T4"
 elif 'A100' in GPU_NAME.upper():
 GPU_MEMORY_GB = 40
 GPU_TYPE = "A100"
 else:
 GPU_MEMORY_GB = 16 # Colab default
 GPU_TYPE = "OTHER"

print(f"\n{'='*60}")
print(f" V11 STABLE GROWTH - ENVIRONMENT SETUP")
print(f"{'='*60}")
print(f" Platform: {'Google Colab' if IN_COLAB else 'Local Machine'}")
print(f" TensorFlow: {tf.__version__}")
print(f" NumPy: {np.__version__}")
print(f" Pandas: {pd.__version__}")
print(f" GPU: {GPU_NAME}")
print(f" GPU Memory: ~{GPU_MEMORY_GB} GB")
print(f"{'='*60}")

In [ ]:
# V11 CONFIGURATION - EVIDENCE-BASED IMPROVEMENTS
# Based on V10 failure analysis:
# - V10 used L1/L2=0.01 (paper spec) → Training AUC stuck at 0.50
# - V9 used L1/L2=0.0001 → AUC 0.738 (best so far)
# - V11 keeps V9's regularization but increases capacity

class Config:
 """
 V11 STABLE GROWTH Configuration
 
 CHANGES FROM V9:
 - Dense units: 1024 → 1536 (50% increase, cautious)
 - Dropout: 0.25 → 0.30 (slight increase for larger model)
 - L2 regularization: 0.0001 → 0.0003 (slight increase)
 
 KEPT FROM V9 (proven to work):
 - CLAHE preprocessing
 - LR warmup
 - Focal loss
 - 3-stage training
 - TTA at inference
 """
 
 def __init__(self, gpu_memory_gb=12):
 self.GPU_MEMORY_GB = gpu_memory_gb
 
 # Detect environment
 try:
 import google.colab
 self.IN_COLAB = True
 except:
 self.IN_COLAB = False
 
 # Set paths
 if self.IN_COLAB:
 self.BASE_PATH = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 self.DICOM_PATH = self.BASE_PATH / 'CBIS-DDSM'
 self.CSV_PATH = self.BASE_PATH / 'csv'
 self.OUTPUT_PATH = self.BASE_PATH / 'model_outputs_v11'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints_v11'
 self.RESULTS_PATH = self.BASE_PATH / 'results_v11'
 # Use V9's cache (same preprocessing)
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v9'
 else:
 self.BASE_PATH = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 self.DICOM_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
 self.CSV_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
 self.OUTPUT_PATH = self.BASE_PATH / 'local_outputs_v11'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'local_checkpoints_v11'
 self.RESULTS_PATH = self.BASE_PATH / 'local_results_v11'
 # Use V9's cache (same preprocessing)
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v9'
 
 # Data files
 self.CALC_TRAIN_CSV = self.CSV_PATH / 'calc_case_description_train_set.csv'
 self.CALC_TEST_CSV = self.CSV_PATH / 'calc_case_description_test_set.csv'
 self.MASS_TRAIN_CSV = self.CSV_PATH / 'mass_case_description_train_set.csv'
 self.MASS_TEST_CSV = self.CSV_PATH / 'mass_case_description_test_set.csv'
 
 # IMAGE CONFIGURATION (same as V9)
 self.IMG_SIZE = (224, 224)
 self.IMG_CHANNELS = 3
 
 # CLAHE PREPROCESSING (same as V9)
 self.USE_CLAHE = True
 self.CLAHE_CLIP_LIMIT = 2.0
 self.CLAHE_TILE_SIZE = (8, 8)
 
 # GPU-OPTIMIZED SETTINGS
 # V11 KEY CHANGE: Stage-specific batch sizes to prevent OOM
 # Stage 3 unfreezes many layers → needs smaller batch
 if gpu_memory_gb >= 16:
 self.BATCH_SIZE = 16
 self.BATCH_SIZE_STAGE3 = 16 # Colab T4 has enough memory
 self.DENSE_UNITS = 2048 # Paper spec possible with high memory
 self.NUM_ENSEMBLE_MODELS = 3
 self.USE_MIXED_PRECISION = True
 self.GPU_PROFILE = "HIGH_MEMORY"
 elif gpu_memory_gb >= 12:
 # TITAN X: Stage 3 needs smaller batch due to unfrozen layers
 self.BATCH_SIZE = 8 # Stages 1 & 2
 self.BATCH_SIZE_STAGE3 = 4 # Stage 3 (more params unfrozen)
 self.DENSE_UNITS = 1536 # V11 CHANGE: 1024 → 1536
 self.NUM_ENSEMBLE_MODELS = 1
 self.USE_MIXED_PRECISION = False
 self.GPU_PROFILE = "TITAN_X"
 else:
 self.BATCH_SIZE = 4
 self.BATCH_SIZE_STAGE3 = 2
 self.DENSE_UNITS = 1024
 self.NUM_ENSEMBLE_MODELS = 1
 self.USE_MIXED_PRECISION = False
 self.GPU_PROFILE = "LOW_MEMORY"
 
 self.ENSEMBLE_SEEDS = [42, 123, 456][:self.NUM_ENSEMBLE_MODELS]
 
 # TEST-TIME AUGMENTATION (same as V9)
 self.USE_TTA = True
 self.TTA_AUGMENTATIONS = 4
 
 # TRAINING PARAMETERS (same as V9)
 self.STAGE1_EPOCHS = 25
 self.STAGE1_LR = 1e-3
 
 self.STAGE2_EPOCHS = 40
 self.STAGE2_LR = 5e-4
 
 self.STAGE3_EPOCHS = 80
 self.STAGE3_LR = 1e-4
 
 self.USE_LR_WARMUP = True
 self.WARMUP_EPOCHS = 5
 self.WARMUP_START_LR = 1e-7
 
 self.TRAIN_SPLIT = 0.70
 self.VAL_SPLIT = 0.15
 self.TEST_SPLIT = 0.15
 
 # REGULARIZATION - V11 CRITICAL PARAMETERS
 # V9: L2=0.0001, dropout=0.25 → AUC 0.738 
 # V10: L2=0.01, dropout=0.5 → AUC 0.640 (model died!)
 # V11: L2=0.0003, dropout=0.30 → slight increase for larger model
 
 self.L1_REGULARIZATION = 0.0 # V11: No L1 (V10's L1=0.01 was harmful)
 self.L2_REGULARIZATION = 3e-4 # V11 CHANGE: 0.0001 → 0.0003 (3x V9)
 self.DROPOUT_RATE = 0.30 # V11 CHANGE: 0.25 → 0.30
 self.LABEL_SMOOTHING = 0.1
 self.GRADIENT_CLIPNORM = 1.0
 
 # CLASS WEIGHTING (same as V9)
 self.USE_CLASS_WEIGHT = True
 self.MALIGNANT_WEIGHT_MULTIPLIER = 2.5
 
 # FOCAL LOSS (same as V9)
 self.USE_FOCAL_LOSS = True
 # ARCHITECTURE - GPU-adaptive freezing
 self.FOCAL_LOSS_GAMMA = 2.0
 
 # Stage 3: Freeze more layers on lower memory GPUs
 if gpu_memory_gb >= 16:
 self.FREEZE_STAGE3 = 100 # Colab T4: can afford more unfrozen
 else:
 self.FREEZE_STAGE3 = 150 # TITAN X: freeze more to save VRAM
 
 # CALLBACKS - V11 IMPROVEMENT
 self.EARLY_STOP_PATIENCE = 25 # V11: Increased from 20
 self.LR_REDUCE_PATIENCE = 10 # V11: Increased from 8
 self.LR_REDUCE_FACTOR = 0.5
 self.MIN_EPOCHS_STAGE1 = 15
 self.MIN_EPOCHS_STAGE2 = 25
 self.MIN_EPOCHS_STAGE3 = 40
 
 # V11 NEW: Training AUC threshold - abort if model is dead
 self.MIN_TRAIN_AUC_THRESHOLD = 0.55
 self.DEAD_MODEL_CHECK_EPOCH = 10
 
 # PERFORMANCE OPTIMIZATION
 self.PREFETCH_BUFFER = tf.data.AUTOTUNE
 self.SHUFFLE_BUFFER = 1000
 self.NUM_PARALLEL_CALLS = tf.data.AUTOTUNE
 self.FORCE_CACHE_REBUILD = False
 
 self._create_dirs()
 
 def _create_dirs(self):
 for path in [self.OUTPUT_PATH, self.CHECKPOINT_PATH, self.RESULTS_PATH]:
 path.mkdir(parents=True, exist_ok=True)
 
 def print_config(self):
 print("\n" + "="*70)
 print(" V11 STABLE GROWTH - EVIDENCE-BASED CONFIGURATION")
 print("="*70)
 
 print(f"\n GPU Profile: {self.GPU_PROFILE} ({self.GPU_MEMORY_GB}GB)")
 print(f" Batch Size: {self.BATCH_SIZE}")
 print(f" Dense Units: {self.DENSE_UNITS} (V9 was 1024)")
 print(f" Ensemble: {self.NUM_ENSEMBLE_MODELS} model(s)")
 
 print(f"\n V11 Key Changes (from V9):")
 print(f" Dense Units: 1024 → {self.DENSE_UNITS} (+50%)")
 print(f" Dropout: 0.25 → {self.DROPOUT_RATE}")
 print(f" L2 Reg: 0.0001 → {self.L2_REGULARIZATION}")
 print(f" Patience: 20 → {self.EARLY_STOP_PATIENCE}")
 
 print(f"\n V10 FAILURE LESSON (what to avoid):")
 print(f" V10 used L1/L2=0.01 → Training AUC stuck at 0.50")
 print(f" V10 AUC: 0.640 (13% worse than V9!)")
 print(f" V11 keeps proven light regularization")
 
 print(f"\n Training Strategy:")
 print(f" Stage 1: {self.STAGE1_EPOCHS} epochs @ LR={self.STAGE1_LR}")
 print(f" Stage 2: {self.STAGE2_EPOCHS} epochs @ LR={self.STAGE2_LR}")
 print(f" Stage 3: {self.STAGE3_EPOCHS} epochs @ LR={self.STAGE3_LR}")
 total_epochs = (self.STAGE1_EPOCHS + self.STAGE2_EPOCHS + self.STAGE3_EPOCHS) * self.NUM_ENSEMBLE_MODELS
 print(f" Total: {total_epochs} epochs")
 
 # Check if cache exists
 cache_files = ['train_cache_v9.npz', 'val_cache_v9.npz', 'test_cache_v9.npz']
 cache_exists = all((self.CACHE_PATH / f).exists() for f in cache_files)
 if cache_exists:
 print(f"\n V9 CACHE FOUND - Reusing preprocessed data!")
 else:
 print(f"\n V9 cache not found - will preprocess")
 
 print("="*70 + "\n")

# Initialize configuration
config = Config(gpu_memory_gb=GPU_MEMORY_GB)
config.print_config()

# Configure GPU
if gpus:
 if config.USE_MIXED_PRECISION:
 try:
 policy = tf.keras.mixed_precision.Policy('mixed_float16')
 tf.keras.mixed_precision.set_global_policy(policy)
 print(" Mixed precision (FP16) enabled")
 except Exception as e:
 print(f" Mixed precision not available: {e}")
 print(" Using FP32 (mixed precision disabled for GPU compatibility)")
 config.USE_MIXED_PRECISION = False
 else:
 print(" Using FP32 (mixed precision disabled for GPU compatibility)")

In [ ]:
# Verify Data Paths & Load V9 Cache
# V11 reuses V9's preprocessed cache (same CLAHE settings)

print(" Verifying data paths and V9 cache status...")
print(f" Environment: {'Colab' if config.IN_COLAB else 'Local'}")

# Check paths
paths_to_check = [
 ('DICOM folder', config.DICOM_PATH),
 ('CSV folder', config.CSV_PATH),
 ('V9 Cache folder', config.CACHE_PATH),
]

all_ok = True
for name, path in paths_to_check:
 if path.exists():
 print(f" {name}")
 else:
 print(f" {name} NOT FOUND: {path}")
 all_ok = False

# Check V9 cache status
cache_files = {
 'train_cache_v9.npz': config.CACHE_PATH / 'train_cache_v9.npz',
 'val_cache_v9.npz': config.CACHE_PATH / 'val_cache_v9.npz',
 'test_cache_v9.npz': config.CACHE_PATH / 'test_cache_v9.npz',
}

print(f"\n V9 Cache Status:")
cache_ready = True
total_cache_mb = 0
for name, path in cache_files.items():
 if path.exists():
 size_mb = path.stat().st_size / (1024 * 1024)
 total_cache_mb += size_mb
 print(f" {name} ({size_mb:.1f} MB)")
 else:
 print(f" {name} - not found")
 cache_ready = False

if cache_ready:
 print(f"\n V9 CACHE READY ({total_cache_mb:.1f} MB total)")
 print(" Will load preprocessed data instantly!")
else:
 print(f"\n V9 cache not found - run V9 notebook first or repreprocess")

if all_ok and cache_ready:
 print(f"\n Ready to proceed!")
else:
 print(f"\n Check configuration")

In [ ]:
# Load V9 Preprocessed Cache
# V11 reuses V9's cache (same CLAHE preprocessing)
# This saves ~10 minutes of preprocessing time!

print("\n" + "="*70)
print(" LOADING V9 PREPROCESSED CACHE")
print("="*70)

# Load cached data
train_cache = np.load(config.CACHE_PATH / 'train_cache_v9.npz')
val_cache = np.load(config.CACHE_PATH / 'val_cache_v9.npz')
test_cache = np.load(config.CACHE_PATH / 'test_cache_v9.npz')

train_images, train_labels = train_cache['images'], train_cache['labels']
val_images, val_labels = val_cache['images'], val_cache['labels']
test_images, test_labels = test_cache['images'], test_cache['labels']

print(f"\n Data loaded from V9 cache:")
print(f" Train: {train_images.shape} ({train_images.nbytes / 1e6:.1f} MB)")
print(f" Val: {val_images.shape} ({val_images.nbytes / 1e6:.1f} MB)")
print(f" Test: {test_images.shape} ({test_images.nbytes / 1e6:.1f} MB)")

print(f"\n Class Distribution:")
for name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
 n_benign = (labels == 0).sum()
 n_malignant = (labels == 1).sum()
 ratio = n_malignant / n_benign if n_benign > 0 else 0
 print(f" {name}: Benign={n_benign}, Malignant={n_malignant} (ratio: 1:{1/ratio:.2f})")

print("="*70)

In [ ]:
# tf.data Pipeline with GPU Augmentation
# Using from_generator to avoid GPU memory errors

@tf.function
def augment_batch(images):
 """V11 GPU-accelerated augmentation (same as V9)."""
 images = tf.image.random_flip_left_right(images)
 images = tf.image.random_flip_up_down(images)
 images = tf.image.random_brightness(images, max_delta=0.1)
 images = tf.image.random_contrast(images, lower=0.9, upper=1.1)
 return images


def create_data_generator(images, labels, shuffle=False, seed=None):
 """Generator function that yields (image, label) pairs."""
 indices = np.arange(len(images))
 if shuffle:
 rng = np.random.default_rng(seed)
 rng.shuffle(indices)
 
 for idx in indices:
 yield images[idx], labels[idx]


def create_dataset(images, labels, config, training=True, shuffle_seed=None):
 """Create optimized tf.data.Dataset using from_generator."""
 output_signature = (
 tf.TensorSpec(shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3), dtype=tf.float32),
 tf.TensorSpec(shape=(), dtype=tf.float32)
 )
 
 dataset = tf.data.Dataset.from_generator(
 lambda: create_data_generator(images, labels, shuffle=training, seed=shuffle_seed),
 output_signature=output_signature
 )
 
 if training:
 dataset = dataset.shuffle(config.SHUFFLE_BUFFER)
 
 dataset = dataset.batch(config.BATCH_SIZE)
 
 if training:
 dataset = dataset.map(
 lambda x, y: (augment_batch(x), y),
 num_parallel_calls=config.NUM_PARALLEL_CALLS
 )
 
 dataset = dataset.prefetch(config.PREFETCH_BUFFER)
 
 return dataset


def create_dataset_staged(images, labels, config, training=True, shuffle_seed=None):
 """
 Create dataset with SMALLER batch size for Stage 3.
 Stage 3 unfreezes many layers, requiring reduced batch to fit in VRAM.
 """
 output_signature = (
 tf.TensorSpec(shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3), dtype=tf.float32),
 tf.TensorSpec(shape=(), dtype=tf.float32)
 )
 
 dataset = tf.data.Dataset.from_generator(
 lambda: create_data_generator(images, labels, shuffle=training, seed=shuffle_seed),
 output_signature=output_signature
 )
 
 if training:
 dataset = dataset.shuffle(config.SHUFFLE_BUFFER)
 
 # Use smaller batch for Stage 3 to prevent OOM
 dataset = dataset.batch(config.BATCH_SIZE_STAGE3)
 
 if training:
 dataset = dataset.map(
 lambda x, y: (augment_batch(x), y),
 num_parallel_calls=config.NUM_PARALLEL_CALLS
 )
 
 dataset = dataset.prefetch(config.PREFETCH_BUFFER)
 
 return dataset



def create_dataset_staged(images, labels, config, training=True, shuffle_seed=None):
 """
 Create dataset with SMALLER batch size for Stage 3.
 Stage 3 unfreezes many layers, requiring reduced batch to fit in VRAM.
 """
 output_signature = (
 tf.TensorSpec(shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3), dtype=tf.float32),
 tf.TensorSpec(shape=(), dtype=tf.float32)
 )
 
 dataset = tf.data.Dataset.from_generator(
 lambda: create_data_generator(images, labels, shuffle=training, seed=shuffle_seed),
 output_signature=output_signature
 )
 
 if training:
 dataset = dataset.shuffle(config.SHUFFLE_BUFFER)
 
 # Use smaller batch for Stage 3 to prevent OOM
 dataset = dataset.batch(config.BATCH_SIZE_STAGE3)
 
 if training:
 dataset = dataset.map(
 lambda x, y: (augment_batch(x), y),
 num_parallel_calls=config.NUM_PARALLEL_CALLS
 )
 
 dataset = dataset.prefetch(config.PREFETCH_BUFFER)
 
 return dataset
# Create datasets
train_dataset = create_dataset(train_images, train_labels, config, training=True)
val_dataset = create_dataset(val_images, val_labels, config, training=False)
test_dataset = create_dataset(test_images, test_labels, config, training=False)

print("\n V11 tf.data pipelines created")
print(f" Batch size (Stage 1/2): {config.BATCH_SIZE}")
print(f" Batch size (Stage 3): {config.BATCH_SIZE_STAGE3}")
print(f" Steps per epoch: {len(train_images) // config.BATCH_SIZE}")

In [ ]:
# Compute Class Weights

balanced_weights = compute_class_weight(
 class_weight='balanced',
 classes=np.array([0, 1]),
 y=train_labels
)

config.CLASS_WEIGHTS = {
 0: balanced_weights[0],
 1: balanced_weights[1] * config.MALIGNANT_WEIGHT_MULTIPLIER
}

print("\n" + "="*70)
print(" V11 CLASS WEIGHTING")
print("="*70)
print(f"\n Training Class Distribution:")
print(f" Benign (0): {int((train_labels == 0).sum())} ({(train_labels == 0).mean()*100:.1f}%)")
print(f" Malignant (1): {int((train_labels == 1).sum())} ({(train_labels == 1).mean()*100:.1f}%)")
print(f"\n Class Weights (with {config.MALIGNANT_WEIGHT_MULTIPLIER}x malignant boost):")
print(f" Benign (0): {config.CLASS_WEIGHTS[0]:.3f}")
print(f" Malignant (1): {config.CLASS_WEIGHTS[1]:.3f}")
print("="*70)

In [ ]:
# Focal Loss Definition

# Add FOCAL_LOSS_ALPHA to config if not present
if not hasattr(config, 'FOCAL_LOSS_ALPHA'):
 config.FOCAL_LOSS_ALPHA = 0.7 # Default alpha for minority class weight

@tf.function
def focal_loss_fn(y_true, y_pred, gamma=2.0, alpha=0.7):
 """Focal loss for handling class imbalance."""
 epsilon = tf.keras.backend.epsilon()
 y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
 y_true = tf.cast(y_true, tf.float32)
 
 ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
 pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
 focal_weight = tf.pow(1 - pt, gamma)
 alpha_weight = y_true * alpha + (1 - y_true) * (1 - alpha)
 
 return tf.reduce_mean(alpha_weight * focal_weight * ce)


def get_focal_loss(gamma, alpha):
 """Return focal loss function for model.compile()."""
 def loss_fn(y_true, y_pred):
 return focal_loss_fn(y_true, y_pred, gamma, alpha)
 return loss_fn

print(" Focal Loss defined")
print(f" α = {config.FOCAL_LOSS_ALPHA} (minority class weight)")
print(f" γ = {config.FOCAL_LOSS_GAMMA} (focusing parameter)")

In [ ]:
# V11 Model Builder - INCREASED CAPACITY
# Key change: Dense units 1024 → 1536 with light regularization

def build_model(config, seed=42):
 """
 Build V11 model with increased capacity.
 
 V11 vs V9:
 - Dense units: 1024 → 1536 (50% increase)
 - L2 regularization: 0.0001 → 0.0003 (slight increase for larger model)
 - Dropout: 0.25 → 0.30 (slight increase)
 
 V11 vs V10 (avoiding the failure):
 - NO L1 regularization (V10 had L1=0.01)
 - Light L2 (0.0003 vs V10's 0.01 - 33x lighter!)
 - Moderate dropout (0.30 vs V10's 0.50)
 """
 tf.random.set_seed(seed)
 np.random.seed(seed)
 
 print(f"\n Building V11 Model (seed={seed})")
 print(f" Dense units: {config.DENSE_UNITS}")
 print(f" L2 regularization: {config.L2_REGULARIZATION}")
 print(f" Dropout: {config.DROPOUT_RATE}")
 
 # Base model
 base_model = DenseNet121(
 include_top=False,
 weights='imagenet',
 input_shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3),
 pooling=None
 )
 base_model.trainable = False
 
 # V11 Classification Head - INCREASED CAPACITY
 x = base_model.output
 x = layers.GlobalAveragePooling2D(name='gap')(x)
 x = layers.BatchNormalization(name='bn_1')(x)
 
 # V11: 1536 units with light L2 regularization
 x = layers.Dense(
 config.DENSE_UNITS,
 activation='relu',
 kernel_regularizer=regularizers.l2(config.L2_REGULARIZATION),
 name=f'dense_{config.DENSE_UNITS}'
 )(x)
 x = layers.BatchNormalization(name='bn_2')(x)
 x = layers.Dropout(config.DROPOUT_RATE, name='dropout')(x)
 
 # Output layer
 output = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)
 
 model = models.Model(inputs=base_model.input, outputs=output, name=f'DenseNet121_V11_seed{seed}')
 
 trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total = model.count_params()
 
 print(f" Total params: {total:,}")
 print(f" Trainable: {trainable:,} ({trainable/total*100:.1f}%)")
 
 return model, base_model

print(" V11 Model builder defined")
print(f" Dense units: {config.DENSE_UNITS} (V9 was 1024)")
print(f" Dropout: {config.DROPOUT_RATE} (V9 was 0.25)")
print(f" L2 regularization: {config.L2_REGULARIZATION} (V9 was 0.0001)")

In [ ]:
# Learning Rate Warmup Scheduler

class WarmupCosineDecayScheduler(Callback):
 """V11: Learning Rate Warmup with Cosine Decay."""
 
 def __init__(self, target_lr, warmup_epochs, total_epochs, 
 warmup_start_lr=1e-7, min_lr=1e-8, verbose=True):
 super().__init__()
 self.target_lr = target_lr
 self.warmup_epochs = warmup_epochs
 self.total_epochs = total_epochs
 self.warmup_start_lr = warmup_start_lr
 self.min_lr = min_lr
 self.verbose = verbose
 
 def on_epoch_begin(self, epoch, logs=None):
 if epoch < self.warmup_epochs:
 lr = self.warmup_start_lr + (self.target_lr - self.warmup_start_lr) * (epoch / self.warmup_epochs)
 else:
 progress = (epoch - self.warmup_epochs) / max(1, (self.total_epochs - self.warmup_epochs))
 lr = self.min_lr + 0.5 * (self.target_lr - self.min_lr) * (1 + np.cos(np.pi * progress))
 
 self.model.optimizer.learning_rate.assign(lr)
 
 if self.verbose and (epoch < self.warmup_epochs or epoch % 10 == 0):
 phase = "warmup" if epoch < self.warmup_epochs else "decay"
 print(f"\n Epoch {epoch+1}: LR = {lr:.2e} ({phase})")


print(" LR Warmup Scheduler defined")
print(f" Warmup epochs: {config.WARMUP_EPOCHS}")

In [ ]:
# V11 Callbacks with Dead Model Detection
# V11 NEW: Monitors training AUC to detect dead models early
# (V10's model was "dead" - train AUC never exceeded 0.53)

class TrainingAUCMonitor(Callback):
 """
 V11 NEW: Monitor training AUC to detect dead models.
 
 If training AUC is too low after N epochs, the model isn't learning.
 This was the key failure signal in V10 that was missed:
 - V10 training AUC never exceeded 0.53 (near random!)
 - V9 training AUC reached 0.70+ early in training
 """
 
 def __init__(self, threshold=0.55, check_after_epoch=10, verbose=True):
 super().__init__()
 self.threshold = threshold
 self.check_after_epoch = check_after_epoch
 self.verbose = verbose
 self.max_train_auc = 0.0
 
 def on_epoch_end(self, epoch, logs=None):
 train_auc = logs.get('auc', 0)
 self.max_train_auc = max(self.max_train_auc, train_auc)
 
 if self.verbose and (epoch + 1) % 5 == 0:
 print(f" Train AUC: {train_auc:.4f} (max so far: {self.max_train_auc:.4f})")
 
 # Check if model is learning
 if epoch >= self.check_after_epoch and self.max_train_auc < self.threshold:
 print(f"\n WARNING: Training AUC ({self.max_train_auc:.4f}) below threshold ({self.threshold})!")
 print(f" This indicates the model may not be learning effectively.")
 print(f" (V10 had this issue - train AUC stuck at 0.50-0.53)")


class DiagnosticCallback(Callback):
 """Lightweight diagnostic callback."""
 
 def __init__(self, check_interval=10):
 super().__init__()
 self.check_interval = check_interval
 
 def on_epoch_end(self, epoch, logs=None):
 if (epoch + 1) % self.check_interval == 0:
 print(f"\n{'='*50}")
 print(f" Epoch {epoch + 1} Checkpoint")
 print(f" Train AUC: {logs.get('auc', 0):.4f}")
 print(f" Val AUC: {logs.get('val_auc', 0):.4f}")
 print(f" Val Loss: {logs.get('val_loss', 0):.4f}")
 print(f"{'='*50}")


def get_callbacks(stage_name, config, model_idx=0, min_epochs=15, 
 use_warmup=False, target_lr=None, total_epochs=None):
 """Get V11 callbacks with dead model detection."""
 
 callbacks = [
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'model{model_idx}_{stage_name}_best_auc.h5'),
 monitor='val_auc',
 mode='max',
 save_best_only=True,
 verbose=1
 ),
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'model{model_idx}_{stage_name}_best_loss.h5'),
 monitor='val_loss',
 mode='min',
 save_best_only=True,
 verbose=0
 ),
 EarlyStopping(
 monitor='val_auc',
 mode='max',
 patience=config.EARLY_STOP_PATIENCE,
 restore_best_weights=True,
 verbose=1,
 start_from_epoch=min_epochs
 ),
 DiagnosticCallback(check_interval=10),
 # V11 NEW: Monitor training AUC
 TrainingAUCMonitor(
 threshold=config.MIN_TRAIN_AUC_THRESHOLD,
 check_after_epoch=config.DEAD_MODEL_CHECK_EPOCH
 )
 ]
 
 if use_warmup and target_lr and total_epochs:
 callbacks.append(WarmupCosineDecayScheduler(
 target_lr=target_lr,
 warmup_epochs=config.WARMUP_EPOCHS,
 total_epochs=total_epochs,
 warmup_start_lr=config.WARMUP_START_LR
 ))
 else:
 callbacks.append(ReduceLROnPlateau(
 monitor='val_loss',
 factor=config.LR_REDUCE_FACTOR,
 patience=config.LR_REDUCE_PATIENCE,
 min_lr=1e-8,
 verbose=1
 ))
 
 return callbacks

print(" V11 Callbacks defined")
print(f" NEW: Training AUC monitor (threshold={config.MIN_TRAIN_AUC_THRESHOLD})")
print(f" Early stopping patience: {config.EARLY_STOP_PATIENCE}")

In [ ]:
# V11 Training Function
# 3-stage progressive unfreezing (same as V9)

def train_single_model(model_idx, seed, config, train_dataset, val_dataset,
 train_images=None, train_labels=None,
 val_images=None, val_labels=None):
 """
 Train V11 model using 3-stage progressive unfreezing.
 
 Stage 1: Head only (25 epochs)
 Stage 2: Last dense block with warmup (40 epochs)
 Stage 3: Partial fine-tune with warmup (80 epochs) - uses smaller batch
 """
 print("\n" + "="*70)
 print(f" TRAINING MODEL {model_idx + 1}/{config.NUM_ENSEMBLE_MODELS} (seed={seed})")
 print("="*70)
 
 # Build model
 model, base_model = build_model(config, seed=seed)
 
 # Get loss function
 if config.USE_FOCAL_LOSS:
 loss_fn = get_focal_loss(config.FOCAL_LOSS_GAMMA, config.FOCAL_LOSS_ALPHA)
 else:
 loss_fn = BinaryCrossentropy(label_smoothing=config.LABEL_SMOOTHING)
 
 metrics = [
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
 
 # Track all histories
 all_history = {'stage1': [], 'stage2': [], 'stage3': []}
 
 # STAGE 1: Train Classification Head Only
 print(f"\n STAGE 1: Training Head ({config.STAGE1_EPOCHS} epochs)")
 print(f" LR: {config.STAGE1_LR}, All backbone frozen")
 
 base_model.trainable = False
 
 optimizer = Adam(learning_rate=config.STAGE1_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
 
 history1 = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE1_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT else None,
 callbacks=get_callbacks('stage1', config, model_idx=model_idx, 
 min_epochs=config.MIN_EPOCHS_STAGE1),
 verbose=1
 )
 
 best_auc_s1 = max(history1.history['val_auc'])
 max_train_auc_s1 = max(history1.history['auc'])
 print(f" Stage 1 best val_auc: {best_auc_s1:.4f}")
 print(f" Stage 1 max train_auc: {max_train_auc_s1:.4f}")
 
 all_history['stage1'] = history1.history
 
 # STAGE 2: Unfreeze Last Dense Block (with warmup)
 print(f"\n STAGE 2: Last Dense Block ({config.STAGE2_EPOCHS} epochs)")
 print(f" LR: {config.STAGE2_LR} with {config.WARMUP_EPOCHS}-epoch warmup")
 
 base_model.trainable = True
 for layer in base_model.layers:
 layer.trainable = False
 
 unfreeze_from = 'conv5_block1'
 found = False
 for layer in base_model.layers:
 if unfreeze_from in layer.name or found:
 layer.trainable = True
 found = True
 
 trainable_count = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 print(f" Trainable params: {trainable_count:,}")
 
 optimizer = Adam(learning_rate=config.WARMUP_START_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
 
 history2 = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE2_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT else None,
 callbacks=get_callbacks('stage2', config, model_idx=model_idx,
 min_epochs=config.MIN_EPOCHS_STAGE2,
 use_warmup=True, target_lr=config.STAGE2_LR,
 total_epochs=config.STAGE2_EPOCHS),
 verbose=1
 )
 
 best_auc_s2 = max(history2.history['val_auc'])
 print(f" Stage 2 best val_auc: {best_auc_s2:.4f}")
 
 all_history['stage2'] = history2.history
 
 # STAGE 3: Partial Fine-Tuning (with warmup)
 # Use smaller batch size for Stage 3 to prevent OOM
 use_smaller_batch = config.BATCH_SIZE_STAGE3 < config.BATCH_SIZE
 
 print(f"\n STAGE 3: Partial Fine-Tune ({config.STAGE3_EPOCHS} epochs)")
 print(f" LR: {config.STAGE3_LR} with {config.WARMUP_EPOCHS}-epoch warmup")
 print(f" Freezing first {config.FREEZE_STAGE3} layers")
 
 if use_smaller_batch and train_images is not None:
 print(f" Using smaller batch: {config.BATCH_SIZE_STAGE3} (was {config.BATCH_SIZE})")
 # Create new datasets with smaller batch for Stage 3
 train_dataset_s3 = create_dataset_staged(
 train_images, train_labels, config, training=True, shuffle_seed=seed)
 val_dataset_s3 = create_dataset_staged(
 val_images, val_labels, config, training=False)
 else:
 train_dataset_s3 = train_dataset
 val_dataset_s3 = val_dataset
 
 # Clear memory before Stage 3
 gc.collect()
 
 base_model.trainable = True
 for i, layer in enumerate(base_model.layers):
 layer.trainable = i >= config.FREEZE_STAGE3
 
 trainable_count = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total_count = model.count_params()
 print(f" Trainable params: {trainable_count:,} ({trainable_count/total_count*100:.1f}%)")
 
 optimizer = Adam(learning_rate=config.WARMUP_START_LR, clipnorm=config.GRADIENT_CLIPNORM)
 model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
 
 history3 = model.fit(
 train_dataset_s3,
 validation_data=val_dataset_s3,
 epochs=config.STAGE3_EPOCHS,
 class_weight=config.CLASS_WEIGHTS if config.USE_CLASS_WEIGHT else None,
 callbacks=get_callbacks('stage3', config, model_idx=model_idx,
 min_epochs=config.MIN_EPOCHS_STAGE3,
 use_warmup=True, target_lr=config.STAGE3_LR,
 total_epochs=config.STAGE3_EPOCHS),
 verbose=1
 )
 
 best_auc_s3 = max(history3.history['val_auc'])
 print(f" Stage 3 best val_auc: {best_auc_s3:.4f}")
 
 all_history['stage3'] = history3.history
 
 # Load best weights
 best_path = config.CHECKPOINT_PATH / f'model{model_idx}_stage3_best_auc.h5'
 if best_path.exists():
 model.load_weights(str(best_path))
 print(f" Loaded best weights from {best_path.name}")
 
 # Summary
 print(f"\n Model {model_idx + 1} Training Summary:")
 print(f" Stage 1: {best_auc_s1:.4f}")
 print(f" Stage 2: {best_auc_s2:.4f}")
 print(f" Stage 3: {best_auc_s3:.4f}")
 print(f" Best: {max(best_auc_s1, best_auc_s2, best_auc_s3):.4f}")
 
 return model, all_history

print(" V11 Training function defined")

In [ ]:
# TRAIN V11 MODEL

print("\n" + "="*70)
print(" V11 STABLE GROWTH TRAINING")
print(f" Training {config.NUM_ENSEMBLE_MODELS} models with seeds: {config.ENSEMBLE_SEEDS}")
total_epochs = (config.STAGE1_EPOCHS + config.STAGE2_EPOCHS + config.STAGE3_EPOCHS) * config.NUM_ENSEMBLE_MODELS
print(f" Total epochs: {total_epochs}")
print("="*70)

# Record start time
training_start = datetime.now()

ensemble_models = []
ensemble_histories = []

for idx, seed in enumerate(config.ENSEMBLE_SEEDS):
 train_ds = create_dataset(train_images, train_labels, config, training=True, shuffle_seed=seed)
 val_ds = create_dataset(val_images, val_labels, config, training=False)
 
 model, histories = train_single_model(
 model_idx=idx,
 seed=seed,
 config=config,
 train_dataset=train_ds,
 val_dataset=val_ds,
 # Pass arrays for Stage 3 dataset recreation with smaller batch
 train_images=train_images,
 train_labels=train_labels,
 val_images=val_images,
 val_labels=val_labels
 )
 
 ensemble_models.append(model)
 ensemble_histories.append(histories)
 
 gc.collect()

training_duration = datetime.now() - training_start

print("\n" + "="*70)
print(f" V11 TRAINING COMPLETE")
print(f" Duration: {training_duration}")
print("="*70)

In [ ]:
# Test-Time Augmentation (TTA)

def apply_tta_augmentation(images, aug_type):
 """Apply specific augmentation for TTA."""
 if aug_type == 0:
 return images
 elif aug_type == 1:
 return np.flip(images, axis=2)
 elif aug_type == 2:
 return np.flip(images, axis=1)
 elif aug_type == 3:
 return np.flip(np.flip(images, axis=2), axis=1)
 return images


def predict_with_tta(model, images, n_augmentations=4, batch_size=32):
 """Make predictions with TTA."""
 all_preds = []
 
 for aug_idx in range(n_augmentations):
 aug_images = apply_tta_augmentation(images, aug_idx)
 preds = model.predict(aug_images, batch_size=batch_size, verbose=0)
 all_preds.append(preds.flatten())
 
 return np.mean(all_preds, axis=0)


def ensemble_predict_with_tta(models, images, config):
 """Make ensemble predictions with TTA."""
 all_model_preds = []
 
 for idx, model in enumerate(models):
 print(f" Predicting with Model {idx + 1}/{len(models)}...")
 
 if config.USE_TTA:
 preds = predict_with_tta(model, images, config.TTA_AUGMENTATIONS, config.BATCH_SIZE)
 else:
 preds = model.predict(images, batch_size=config.BATCH_SIZE, verbose=0).flatten()
 
 all_model_preds.append(preds)
 
 ensemble_preds = np.mean(all_model_preds, axis=0)
 
 return ensemble_preds, all_model_preds

print(" TTA functions defined")
print(f" TTA augmentations: {config.TTA_AUGMENTATIONS}")

In [ ]:
# Final Evaluation

print("\n" + "="*70)
print(" V11 FINAL EVALUATION")
print("="*70)

# Get predictions
y_pred_ensemble, individual_preds = ensemble_predict_with_tta(ensemble_models, test_images, config)
y_true = test_labels

# Calculate AUC for each model
print("\n Individual Model AUCs:")
aucs = []
for idx, preds in enumerate(individual_preds):
 model_auc = roc_auc_score(y_true, preds)
 aucs.append(model_auc)
 print(f" Model {idx + 1}: {model_auc:.4f}")

# Ensemble AUC
auc = roc_auc_score(y_true, y_pred_ensemble)
print(f"\n Ensemble AUC: {auc:.4f}")

# Find optimal threshold
fpr, tpr, thresholds = roc_curve(y_true, y_pred_ensemble)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]
print(f" Optimal threshold: {optimal_threshold:.4f}")

# Apply threshold
y_pred_opt = (y_pred_ensemble >= optimal_threshold).astype(int)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_opt)
tn, fp, fn, tp = cm.ravel()

# Calculate metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\n V11 Test Metrics:")
print(f" AUC: {auc:.4f}")
print(f" Sensitivity: {sensitivity:.4f} ({sensitivity*100:.1f}%)")
print(f" Specificity: {specificity:.4f} ({specificity*100:.1f}%)")
print(f" Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f" PPV: {ppv:.4f}")
print(f" NPV: {npv:.4f}")
print(f" FNR: {fnr:.4f}")

print(f"\n Confusion Matrix:")
print(f" TP={tp}, TN={tn}, FP={fp}, FN={fn}")

In [ ]:
# Compare with Previous Versions

print("\n" + "="*70)
print(" V11 vs PREVIOUS VERSIONS")
print("="*70)

comparison = {
 'V4': {'auc': 0.707, 'sensitivity': 0.583, 'specificity': 0.762},
 'V6': {'auc': 0.723, 'sensitivity': 0.633, 'specificity': 0.698},
 'V7': {'auc': 0.732, 'sensitivity': 0.697, 'specificity': 0.660},
 'V9': {'auc': 0.738, 'sensitivity': 0.587, 'specificity': 0.762},
 'V10': {'auc': 0.640, 'sensitivity': 0.647, 'specificity': 0.584},
 'V11': {'auc': auc, 'sensitivity': sensitivity, 'specificity': specificity}
}

print(f"\n{'Version':<10} {'AUC':<10} {'Sensitivity':<15} {'Specificity':<15}")
print("-" * 50)
for version, metrics in comparison.items():
 marker = " ← CURRENT" if version == 'V11' else (" ← BEST" if version == 'V9' else "")
 marker = " ← FAILED" if version == 'V10' else marker
 print(f"{version:<10} {metrics['auc']:<10.4f} {metrics['sensitivity']*100:<15.1f}% {metrics['specificity']*100:<15.1f}%{marker}")

# Calculate improvements
auc_vs_v9 = auc - 0.738
auc_vs_v10 = auc - 0.640

print(f"\n V11 Improvement:")
print(f" vs V9: {auc_vs_v9:+.4f} AUC ({auc_vs_v9/0.738*100:+.1f}%)")
print(f" vs V10: {auc_vs_v10:+.4f} AUC ({auc_vs_v10/0.640*100:+.1f}%)")

In [ ]:
# Save Results and Plot

# Prepare results dictionary
results = {
 'version': 'V11_STABLE_GROWTH',
 'timestamp': datetime.now().isoformat(),
 'training_duration': str(training_duration),
 'configuration': {
 'dense_units': config.DENSE_UNITS,
 'batch_size': config.BATCH_SIZE,
 'l1_regularization': config.L1_REGULARIZATION,
 'l2_regularization': config.L2_REGULARIZATION,
 'dropout_rate': config.DROPOUT_RATE,
 'use_clahe': config.USE_CLAHE,
 'use_lr_warmup': config.USE_LR_WARMUP,
 'use_tta': config.USE_TTA,
 'tta_augmentations': config.TTA_AUGMENTATIONS,
 'img_size': list(config.IMG_SIZE),
 'gpu_profile': config.GPU_PROFILE
 },
 'test_metrics': {
 'auc_roc': float(auc),
 'optimal_threshold': float(optimal_threshold),
 'sensitivity': float(sensitivity),
 'specificity': float(specificity),
 'accuracy': float(accuracy),
 'ppv': float(ppv),
 'npv': float(npv)
 },
 'confusion_matrix': {
 'tp': int(tp),
 'tn': int(tn),
 'fp': int(fp),
 'fn': int(fn)
 },
 'comparison': comparison,
 'individual_model_aucs': [float(a) for a in aucs],
 'improvement_vs_v9': {
 'auc_change': float(auc - 0.738),
 'sensitivity_change': float(sensitivity - 0.587),
 'specificity_change': float(specificity - 0.762)
 },
 'training_history': {
 f'model_{i}': h for i, h in enumerate(ensemble_histories)
 }
}

# Save results
results_file = config.RESULTS_PATH / 'v11_results.json'
with open(results_file, 'w') as f:
 json.dump(results, f, indent=2, default=str)
print(f" Results saved to {results_file}")

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROC Curve
ax1 = axes[0, 0]
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'V11 (AUC={auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax1.scatter(fpr[optimal_idx], tpr[optimal_idx], s=100, c='red', zorder=5, 
 label=f'Optimal (threshold={optimal_threshold:.3f})')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('V11 ROC Curve')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# 2. Confusion Matrix
ax2 = axes[0, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
 xticklabels=['Benign', 'Malignant'],
 yticklabels=['Benign', 'Malignant'])
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
ax2.set_title('V11 Confusion Matrix')

# 3. Version Comparison
ax3 = axes[1, 0]
versions = list(comparison.keys())
auc_values = [comparison[v]['auc'] for v in versions]
colors = ['green' if v == 'V11' else ('red' if v == 'V10' else 'blue') for v in versions]
bars = ax3.bar(versions, auc_values, color=colors, alpha=0.7)
ax3.axhline(y=0.738, color='orange', linestyle='--', label='V9 Baseline')
ax3.set_ylabel('AUC-ROC')
ax3.set_title('AUC Comparison Across Versions')
ax3.set_ylim(0.5, 0.85)
ax3.legend()

# Add value labels
for bar, val in zip(bars, auc_values):
 ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# 4. Training History (Stage 3 val_auc)
ax4 = axes[1, 1]
for i, hist in enumerate(ensemble_histories):
 if 'stage3' in hist and 'val_auc' in hist['stage3']:
 ax4.plot(hist['stage3']['val_auc'], label=f'Model {i+1} Val AUC', alpha=0.8)
 ax4.plot(hist['stage3']['auc'], '--', label=f'Model {i+1} Train AUC', alpha=0.5)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('AUC')
ax4.set_title('V11 Stage 3 Training History')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()

# Save figure
fig_path = config.RESULTS_PATH / 'v11_results.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f" Figure saved to {fig_path}")

plt.show()

In [ ]:
# Final Summary

print("\n" + "="*70)
print(" V11 STABLE GROWTH - FINAL SUMMARY")
print("="*70)

print(f"\n Key Results:")
print(f" AUC: {auc:.4f}")
print(f" Sensitivity: {sensitivity*100:.1f}%")
print(f" Specificity: {specificity*100:.1f}%")
print(f" Accuracy: {accuracy*100:.1f}%")

print(f"\n V11 Configuration:")
print(f" Dense Units: {config.DENSE_UNITS} (V9 was 1024)")
print(f" L2 Reg: {config.L2_REGULARIZATION} (V9 was 0.0001)")
print(f" Dropout: {config.DROPOUT_RATE} (V9 was 0.25)")

print(f"\n Comparison:")
print(f" vs V9: {auc - 0.738:+.4f} AUC")
print(f" vs V10: {auc - 0.640:+.4f} AUC")

print(f"\n Outputs:")
print(f" Results: {results_file}")
print(f" Figure: {fig_path}")
print(f" Checkpoints: {config.CHECKPOINT_PATH}")

print(f"\n⏱ Training Duration: {training_duration}")

if auc > 0.738:
 print(f"\n SUCCESS: V11 beats V9 baseline!")
elif auc > 0.640:
 print(f"\n RECOVERY: V11 beats V10's failed run")
else:
 print(f"\n Need further investigation")

print("\n" + "="*70)